In [0]:
CREATE OR REPLACE TABLE data_governance.gold_access.dim_users USING DELTA AS

SELECT DISTINCT
    COALESCE(a.run_by, u.initiated_by) AS user_id,
    a.account_id,
    a.workspace_id,
    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_access_management.audit_logs a
FULL OUTER JOIN data_governance.silver_access_management.assistant_usage u
    ON a.workspace_id = u.workspace_id

In [0]:
CREATE OR REPLACE TABLE data_governance.gold_access.dim_table_permissions
USING DELTA AS
SELECT
  grantor,
  grantee,
  catalog_name,
  schema_name,
  table_name,
  privilege_type
  is_grantable,
  inherited_from,
  is_grantable_flag,
  full_table_path,
  permission_category,
  CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_access_management.table_permissions

In [0]:
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_access.fact_assistant_uses
),

new_audit AS (
    SELECT *
    FROM data_governance.silver_access_management.audit_logs
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

new_assistant AS (
    SELECT *
    FROM data_governance.silver_access_management.assistant_usage
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

combined AS (
    SELECT
        COALESCE(a.account_id, u.account_id) AS account_id,
        COALESCE(a.workspace_id, u.workspace_id) AS workspace_id,

        COALESCE(a.run_by, u.initiated_by) AS user_id,

        COALESCE(a.event_id, u.event_id) AS event_id,
        COALESCE(a.event_time, u.event_time) AS event_time,
        COALESCE(a.event_date, u.event_date) AS event_date,

        a.service_name,
        a.action_name,

        u.user_agent,
        u.browser,
        u.os,
        u.is_bot,

        a.session_id,
        a.source_ip_address,

        CURRENT_TIMESTAMP() AS load_timestamp

    FROM new_audit a
    FULL OUTER JOIN new_assistant u
        ON a.workspace_id = u.workspace_id
        AND a.event_date = u.event_date
)

INSERT INTO data_governance.gold_access.fact_assistant_uses
SELECT * FROM combined;

In [0]:
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_access.fact_user_access_activity
),

-- ✅ STEP 1: NEW DATA FROM SILVER
new_audit AS (
    SELECT *
    FROM data_governance.silver_access_management.audit_logs
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

-- ✅ STEP 2: DEDUPLICATION
dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY load_timestamp DESC) rn
        FROM new_audit
    ) t
    WHERE rn = 1
),

-- ✅ STEP 3: PASS THROUGH
filtered AS (
    SELECT *
    FROM dedup
),

-- ✅ STEP 4: FINAL (NO TRANSFORMATIONS, ONLY COLUMN SELECTION)
final_data AS (
    SELECT
        -- EVENT
        event_id,
        event_timestamp,
        event_time,
        event_date,
        version,
        audit_level,

        -- ORG
        account_id,
        workspace_id,
        workspace_id_req,
        metastore_id,

        -- USER
        user_email,
        user_subject_name,
        run_by,
        run_as,
        run_by_display_name,
        run_as_display_name,
        is_system_user,

        -- ACCESS
        service_name,
        action_name,
        catalog_name,
        schema_name,
        acting_resource,
        include_browse,

        -- TECH
        client_type,
        source_ip_address,
        is_internal_ip,
        user_agent,
        session_id,

        -- RESULT
        status_code,
        error_message,
        response_result,
        is_success,
        max_results,
        request_id,

        -- PARTITION
        event_year,
        event_month,
        event_day,
        event_hour,

        -- PIPELINE (keep as-is from silver)
        CURRENT_TIMESTAMP() AS load_timestamp
    FROM filtered
)

-- ✅ FINAL INSERT (EXPLICIT ORDER MATCHES TABLE)
INSERT INTO data_governance.gold_access.fact_user_access_activity
SELECT
    event_id,
    event_timestamp,
    event_time,
    event_date,
    version,
    audit_level,
    account_id,
    workspace_id,
    workspace_id_req,
    metastore_id,
    user_email,
    user_subject_name,
    run_by,
    run_as,
    run_by_display_name,
    run_as_display_name,
    is_system_user,
    service_name,
    action_name,
    catalog_name,
    schema_name,
    acting_resource,
    include_browse,
    client_type,
    source_ip_address,
    is_internal_ip,
    user_agent,
    session_id,
    status_code,
    error_message,
    response_result,
    is_success,
    max_results,
    request_id,
    event_year,
    event_month,
    event_day,
    event_hour,
    load_timestamp
FROM final_data;